# ChromaDB — CRUD Operations

ChromaDB supports full CRUD — you can add, read, update, and delete individual documents without rebuilding the collection from scratch. All operations happen on a collection object.

---

### Create — `.add()`

Adds new documents to the collection. ChromaDB auto-generates embeddings.

```python
collection.add(
    documents=["This is a document about LangChain"],
    metadatas=[{"source": "langchain.com"}],
    ids=["id1"]
)
```

---

### Read — `.get()`

Retrieves documents from the collection. Without any arguments, returns everything. You can filter by IDs, metadata, or document content.

```python
# Get all documents
all_items = collection.get()

# Get specific documents by ID
specific = collection.get(ids=["id1", "id2"])

# Get with metadata filter
filtered = collection.get(
    where={"source": "langchain.com"}
)

# Get with document content filter
content_filtered = collection.get(
    where_document={"$contains": "LangChain"}
)
```

`.get()` returns a dictionary with keys `ids`, `documents`, `metadatas`, `embeddings`.

---

### Update — `.update()`

Updates existing documents (by ID). You can update the document text, metadata, or both. ChromaDB re-generates the embedding automatically if you change the document text.

```python
# Update document text and metadata
collection.update(
    ids=["id1"],
    documents=["Updated document about LangChain v2"],
    metadatas=[{"source": "langchain.com", "version": 2.0}]
)
```

**Note:** the ID must already exist in the collection — `.update()` won't create a new document if the ID doesn't exist. For that, use `.upsert()`.

---

### Upsert — `.upsert()`

**Update if exists, insert if not** — safer than `.update()` when you're not sure if a document already exists.

```python
collection.upsert(
    ids=["id1", "id99"],
    documents=["Updated doc", "Brand new doc"],
    metadatas=[{"source": "a.com"}, {"source": "b.com"}]
)
```

`id1` gets updated, `id99` gets created — all in one call.

---

### Delete — `.delete()`

Deletes documents by ID, metadata filter, or document content filter.

```python
# Delete by ID
collection.delete(ids=["id1"])

# Delete by metadata filter
collection.delete(
    where={"source": "langchain.com"}
)

# Delete by document content
collection.delete(
    where_document={"$contains": "LangChain"}
)
```

---

### Collection-Level Operations

Beyond document-level CRUD, ChromaDB also lets you manage collections themselves.

---

```python
# List all collections in the database
client.list_collections()
```

---

```python
# Get an existing collection (instead of creating a new one)
collection = client.get_collection(
    name="my_collection",
    embedding_function=ef
)
```

"Get" here means **fetching/accessing an already existing collection** so you can work with it — not creating anything new.

Think of it like this: `create_collection()` is like creating a new folder. `get_collection()` is like opening an existing folder that's already there.

> Note that any operation you perform on the reference **directly affects the original collection**. It's not a copy, it's literally a reference pointing to the same collection in the database.

#### Practical scenario

Say you ran your script yesterday, created `"my_grocery_collection"`, and added 100 documents to it. Today you open a new script/session — that collection still exists in the database. But your new Python session has no reference to it yet.

```python
# This would FAIL — collection already exists, can't create it again
collection = client.create_collection(name="my_grocery_collection")

# This is CORRECT — just grab the existing one
collection = client.get_collection(
    name="my_grocery_collection",
    embedding_function=ef
)
```

Now `collection` points to that existing collection with all its 100 documents still intact — you didn't touch the data, you just got a reference to it so you can call `.add()`, `.query()`, `.delete()` etc. on it again.

**Why pass `embedding_function` here?** ChromaDB stores the vectors but doesn't store which embedding function you used. So when you re-open a collection, you need to tell it again which model to use for embedding new queries/documents — otherwise it won't know how to embed your search queries.

---

```python
# Get or create — creates if doesn't exist, gets if it does
collection = client.get_or_create_collection(
    name="my_collection",
    embedding_function=ef
)
```

---

```python
# Delete an entire collection
client.delete_collection(name="my_collection")
```

---

```python
# Count documents in a collection
collection.count()
```

---

### Quick Reference

| Operation | Method | What it does |
|---|---|---|
| Add | `.add()` | Insert new documents |
| Read | `.get()` | Retrieve documents |
| Update | `.update()` | Update existing documents (ID must exist) |
| Upsert | `.upsert()` | Update if exists, insert if not |
| Delete | `.delete()` | Remove documents |
| Count | `.count()` | Number of documents in collection |
| List collections | `client.list_collections()` | All collections in the DB |
| Delete collection | `client.delete_collection()` | Remove entire collection |